Ask an LLM to mark the 3-way task, optionally based on training data or ILP prediction.

In [1]:
# Only run this cell if you haven't installed the requirements yet:
%pip install -r requirements-llm.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:

tasks = ["2-way", "3-way"] 
tracks = ["unseen_answers"]#, "unseen_questions"]
# "gold" and "ilponly" cases generated elsewhere from *_TASK_with_prediction.json
#cases = ["trained", "with_ilp", "with_theory", "with_test_ilp", "with_ilp_and_theory", "with_test_ilp_and_theory"] #
cases = ["trained", "with_theory"]#, "with_test_ilp", "with_test_ilp_and_theory"] 


################################################################################
# These are the main parameters to set for the experiment. Adjust as needed.   #
# What are we doing?                                                           #
################################################################################
#track = tracks[0]
#task = tasks[0]
#case = cases[0] 

In [3]:
schema = {
    "3-way": {
        "type": "object",
        "additionalProperties": False,
        "required": ["id", "question_id", "score"],
        "properties": {
            "id": {
                "type": "string"
            },
            "question_id": {
                "type": "string"
            },
            "score": {
                "enum": ["Incorrect", "Partially correct", "Correct"]
            },
            "explanation": {
                "type": "string"
            }
        }
    },
    "2-way": {
        "type": "object",
        "additionalProperties": False,
        "required": ["id", "question_id", "score"],
        "properties": {
            "id": {
                "type": "string"
            },
            "question_id": {
                "type": "string"
            },
            "score": {
                "enum": ["Incorrect", "Correct"]
            },
            "explanation": {
                "type": "string"
            }
        }
    }
}


In [4]:
import pandas as pd
import ujson

from dotenv import load_dotenv
load_dotenv()

import os

cwd = os.getcwd()
print(f"Current working directory: {cwd}")

from mistralai.client import Mistral

import random

import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

client = Mistral(api_key=os.getenv("MISTRALAI_API_KEY"))

Current working directory: /Users/at6737/Work/Projects/AI & Assessment/BEA 2026/bea-2026


In [ ]:
# OneMassiveFunctionTM because this is how smart people do it, right?
# Copilot wrote all of the next comment:
#  Right?? (Also, I want to be able to easily run different cases without having to copy-paste a bunch of code, and I don't want to have to worry about variable scope issues with global variables, so here we are.)
def run_experiment(task, track, case):
    # Include ILP theory in the prompt 
    if "theory" in case:
        include_theory=True
    else:
        include_theory=False
    # Whether to include ILP results for the test question in the prompt (only relevant if arm == "with_ilp")
    if "test_ilp" in case:
        with_test_ilp = True
    else:
        with_test_ilp = False

    #inputfile = os.path.join(f"{task}", f"trial_{task.replace('-', '')}_with_prediction.json")
    inputfile = os.path.join(f"BEA_full_challenge_attempt/{task}-test", f"{task.replace('-', '')}_{track}_with_ilp_eval.json")
    trainingfile = os.path.join(f"{task}", f"train_{task.replace('-', '')}_with_prediction.json")
    theoryfile = os.path.join(f"{task}", f"{task}_theories.json")
    promptprefix = f"{task}_trained" if not "ilp" in case else f"{task}_with_ilp"
    promptfile = os.path.join(f"{task}", "prompts", f"{promptprefix}.txt")
    theoryprompt = os.path.join(f"{task}", "prompts", f"theory_prompt_fragment.txt")
    responsesprefix = f"{task}_{case}_responses"
    responsesfile = os.path.join(".", "results", f"{responsesprefix}.json")


    # Read in prompts
    with open(promptfile, "r") as f:
        prompt = f.read()

    splitprompt = prompt.split("#####")
    systemprompt = splitprompt[0].replace("${task}", str(schema[task]))
    prompt = splitprompt[1]

    if include_theory:
        with open(theoryprompt, "r") as f:
            theory_fragment = f.read()
        systemprompt += "\n\n" + theory_fragment

    # Read in training data
    with open(trainingfile, "r") as f:
        training_data = ujson.load(f)
    # Read in input data
    with open(inputfile, "r") as f:
        input_data = ujson.load(f)

    count = 0
    responses = []
    #batch_requests = []
    # Process each input item in turn
    for current in input_data:
        # Limit results for debugging
        # if count > 5:
        #    break
        count += 1
        current_training_data = []
        for item in training_data:
            if item["question_id"] == current["question_id"]:
                current_training_data.append(item)
        current_system_prompt = systemprompt.replace(
            "${training}", str(current_training_data))
        if include_theory:
            with open(theoryfile, "r") as f:
                theory_data = ujson.load(f)
            current_system_prompt = current_system_prompt.replace("${theory}", str(
                theory_data.get(current["question_id"], "No theory available")))
        current.pop("score", None)
        if "with_ilp" in case and not with_test_ilp:
            current.pop("amati", None)
        current_prompt = prompt.replace("${data}", str(current))
        print(f"Current: {count}/{len(input_data)}")
        # full_current_prompt_tokenized = tokenizer.encode(current_system_prompt+current_prompt)
        # print(len(full_current_prompt_tokenized))

        response = client.chat.complete(
            model="mistral-large-latest",
            messages=[
                {
                    "role": "system",
                    "content": current_system_prompt
                },
                {
                    "role": "user",
                    "content": current_prompt
                }
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "bea",
                    "strict": True,
                    "schema_definition": schema[task]
                }
            }
        )
        #batch_request = {
        #    "custom_id": f"{case}_{count}",
        #    "body": {
        #        "messages": messages,
        #        "response_format": response_format
        #    }
        #}
        #batch_requests.append(batch_request)

    #with open(os.path.join("results", f"batch-{task}-{case}.jsonl"), "w") as f:
    #    for request in batch_requests:
    #        f.write(ujson.dumps(request) + "\n")

        responses.append(response.choices[0].message.content)
        #print(response.choices[0].message.content)

    properresponses = []
    for entry in responses:
        properresponses.append(ujson.loads(entry))
    with open(responsesfile, "w") as f:
        f.write("{\n")
        f.write(f"    \"task\": \"{task}\",\
        \"case\": \"{case}\",\
        \"track\": \"{track}\",\
        \"results\": ")
        ujson.dump(properresponses, f)
        f.write("}")

In [12]:
for track in tracks:
    for task in tasks:
        for case in cases:
            if task == "2-way" and case == "with_theory": # We've done it already
                continue
            run_experiment(task, track, case)

Current: 1/2008
29
Current: 2/2008
29
Current: 3/2008
29
Current: 4/2008
29
Current: 5/2008
29
Current: 6/2008
29
Current: 7/2008
29
Current: 8/2008
29
Current: 9/2008
29
Current: 10/2008
124
Current: 11/2008
124
Current: 12/2008
124
Current: 13/2008
124
Current: 14/2008
124
Current: 15/2008
124
Current: 16/2008
124
Current: 17/2008
124
Current: 18/2008
124
Current: 19/2008
124
Current: 20/2008
124
Current: 21/2008
124
Current: 22/2008
124
Current: 23/2008
124
Current: 24/2008
124
Current: 25/2008
124
Current: 26/2008
124
Current: 27/2008
124
Current: 28/2008
124
Current: 29/2008
124
Current: 30/2008
124
Current: 31/2008
124
Current: 32/2008
124
Current: 33/2008
124
Current: 34/2008
124
Current: 35/2008
124
Current: 36/2008
124
Current: 37/2008
124
Current: 38/2008
124
Current: 39/2008
124
Current: 40/2008
124
Current: 41/2008
124
Current: 42/2008
124
Current: 43/2008
124
Current: 44/2008
124
Current: 45/2008
30
Current: 46/2008
30
Current: 47/2008
30
Current: 48/2008
30
Current: 49/20

In [9]:
print(f"{task} {case}")

3-way with_theory
